# Self-reflection as Monte Carlo — interactive companion

Companion to [Post 4b: Self-Reflection as Monte Carlo](../posts/04b-self-reflection.qmd).

Self-reflection is generate → critique → revise, run by the *same* model.
This notebook lets you watch the fix-vs-break balance tip, find the
discrimination threshold below which reflection hurts, see over-correction
happen round by round, and confirm external feedback fixes everything.

**You'll do (~20 minutes):**
1. Reproduce the closed-form Δ = (1−a)·d·a′ − a·f·(1−a′).
2. Find the d > f threshold where reflection flips from helping to hurting.
3. Watch over-correction erode correct answers.
4. Replace self-critique with an oracle and recover the geometric ceiling.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 110

from nano_agents.reflection import (
    ReflectiveQA, accuracy, single_shot, reflect, reflect_with_oracle,
    best_of_n, self_consistency, one_round_accuracy, reflection_delta,
)

rng = np.random.default_rng(0)

## 1. The two-line result: fix minus break

One round of reflection changes accuracy by

    Δ = (1−a)·d·a′   (FIX: catch a wrong answer, revise it)
      − a·f·(1−a′)   (BREAK: false-alarm a right answer, botch it)

With blind revision (a′ = a) this is Δ = a(1−a)(d−f). Let's confirm the
simulation matches the formula.

In [ ]:
a, d, f = 0.4, 0.8, 0.2
env = ReflectiveQA(n_problems=4000, gen_accuracy=a, detect_rate=d,
                   false_alarm=f, revise_gain=0.0, seed=0)

base = accuracy(env, single_shot, n_trials=3, seed=1)
one_round = accuracy(env, reflect, n_trials=3, seed=1, max_rounds=1)

print(f"single-shot accuracy        : {base:.3f}")
print(f"after 1 round (simulated)   : {one_round:.3f}")
print(f"after 1 round (analytic)    : {one_round_accuracy(a, d, f, a):.3f}")
print(f"Δ analytic  a(1-a)(d-f)     : {a*(1-a)*(d-f):+.3f}")
print(f"Δ simulated                 : {one_round - base:+.3f}")

### Try this
- Flip the critic to over-critical: `d=0.2, f=0.8`. Δ goes negative — reflection
  *lowers* accuracy by the same magnitude.
- Set `d == f`. Δ ≈ 0: an uninformative critic does nothing, forever.

## 2. The discrimination threshold

Reflection helps iff the critic discriminates (d > f). Sweep detection at
fixed false-alarm and watch the gain cross zero exactly at d = f.

In [ ]:
a, f = 0.4, 0.2
detects = np.linspace(0.0, 1.0, 21)
base = None
deltas = []
for d in detects:
    env = ReflectiveQA(n_problems=1500, gen_accuracy=a, detect_rate=float(d),
                       false_alarm=f, revise_gain=0.0, seed=0)
    if base is None:
        base = accuracy(env, single_shot, n_trials=4, seed=1)
    deltas.append(accuracy(env, reflect, n_trials=4, seed=1, max_rounds=1) - base)

disc = detects - f
plt.axhline(0, color="#888", lw=1); plt.axvline(0, color="#888", ls=":", lw=1.5)
plt.plot(disc, a*(1-a)*disc, "-", color="#3a7ebf", lw=2, label="analytic a(1-a)(d-f)")
plt.plot(disc, deltas, "o", color="#c44e52", label="simulated")
plt.xlabel("critic discrimination d − f"); plt.ylabel("accuracy change")
plt.title("Reflection helps iff d > f"); plt.legend(); plt.grid(alpha=0.3); plt.show()

The line passes through the origin: a critic that can't beat chance at
telling right from wrong cannot help, no matter how you prompt it.

## 3. Over-correction: watch correct answers erode

With an over-critical critic (f > d), iterating *lowers* accuracy. Track it
round by round, split by whether the first answer was correct.

In [ ]:
a = 0.4
env = ReflectiveQA(n_problems=1500, gen_accuracy=a, detect_rate=0.4,
                   false_alarm=0.55, revise_gain=0.0, seed=0)
rounds = list(range(0, 9))
ys = [accuracy(env, reflect, n_trials=5, seed=1, max_rounds=r) for r in rounds]

plt.plot(rounds, ys, "o-", color="#c44e52", lw=2, label="reflection")
plt.axhline(a, color="#888", ls="--", lw=1.5, label=f"single-shot ({a})")
plt.xlabel("rounds"); plt.ylabel("accuracy")
plt.title("Over-correction: more rounds, lower accuracy"); plt.legend()
plt.grid(alpha=0.3); plt.show()
print(f"0 rounds: {ys[0]:.3f}   8 rounds: {ys[-1]:.3f}  (it got WORSE)")

### Try this
- Restore a good critic (`detect_rate=0.85, false_alarm=0.1`). Now more rounds
  *help* and plateau — diminishing returns instead of erosion.

## 4. External feedback fixes everything

Replace self-critique with a perfect external verifier (a unit test, a
compiler). It never false-alarms, so accuracy climbs toward the geometric
ceiling 1 − (1−a)^(r+1).

In [ ]:
a = 0.35
env = ReflectiveQA(n_problems=1500, gen_accuracy=a, detect_rate=0.7,
                   false_alarm=0.25, revise_gain=0.0, seed=0)
rounds = list(range(0, 9))
self_crit = [accuracy(env, reflect, n_trials=5, seed=1, max_rounds=r) for r in rounds]
oracle = [accuracy(env, reflect_with_oracle, n_trials=5, seed=1, max_rounds=r) for r in rounds]

plt.plot(rounds, oracle, "o-", color="#3a7ebf", lw=2, label="external verifier")
plt.plot(rounds, [1-(1-a)**(r+1) for r in rounds], ":", color="#3a7ebf",
         label="theory 1-(1-a)^(r+1)")
plt.plot(rounds, self_crit, "s-", color="#c44e52", lw=2, label="pure self-critique")
plt.axhline(a, color="#888", ls="--", lw=1.5, label=f"single-shot ({a})")
plt.xlabel("rounds"); plt.ylabel("accuracy")
plt.title("External feedback vs self-critique"); plt.legend(loc="lower right")
plt.grid(alpha=0.3); plt.ylim(0, 1.02); plt.show()

The gap between the curves is the cost of trusting self-judgement instead of
checking the world. Whenever a cheap verifier exists, wire it in.

## What's next

You've seen self-reflection as adaptive Monte Carlo:

- **Δ = fix − break** — reflection helps iff the critic discriminates (d > f).
- **Helps most when mostly wrong** — and shrinks to nothing as accuracy → 1.
- **Over-correction** — an over-critical critic erodes correct answers.
- **External feedback** — a real verifier turns reflection into a reliable
  error-killer; pure self-critique plateaus.

The next post — [Post 4c: Calibration and uncertainty](../posts/04c-calibration.qmd) —
asks the question all of this depends on: when the model says it's confident,
is it right? Confidence-gated retrieval (4a) and "reflect on the hard ones"
(4b) both need a model that knows what it doesn't know.